In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

SST_FILE = "sst_eof_svd_results.nc"
PR_TRAIN_FILE = "mswep_brunei_landmask_PC_train.nc"
PR_TEST_FILE = "mswep_brunei_landmask_PC_test.nc"

N_SST_MODES = 6
N_PR_MODES = 3

TRAIN_END = pd.Period("2020-12", freq="M")
TEST_START = pd.Period("2021-01", freq="M")
TEST_END = pd.Period("2025-12", freq="M")

OUTPUT_DIR = Path("ml_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [38]:
# 1. LOAD SST PC TRAIN AND TEST

sst_ds = xr.open_dataset(SST_FILE)
sst_train = sst_ds["PC_train"].to_pandas()
sst_test = sst_ds["PC_test"].to_pandas()

# Standardize time index to monthly periods.
sst_train.index = pd.PeriodIndex(
    pd.to_datetime(sst_train.index), freq="M"
)
sst_test.index = pd.PeriodIndex(
    pd.to_datetime(sst_test.index), freq="M"
)

sst_pc = pd.concat([sst_train, sst_test]).sort_index()

# Keep the first six SST PCs.
sst_pc = sst_pc.iloc[:, :N_SST_MODES].copy()
sst_pc.columns = [
    f"sst_pc{mode}" for mode in range(1, N_SST_MODES + 1)
]

if sst_pc.index.has_duplicates:
    raise ValueError("Duplicate months found in SST PC data.")

print("SST PC period:", sst_pc.index.min(), "—", sst_pc.index.max())
print("SST PC shape:", sst_pc.shape)

SST PC period: 1981-09 — 2025-12
SST PC shape: (532, 6)


In [39]:
# 2. LOAD PRECIPITATION PC TRAIN AND TEST

def load_pc_file(path):
    ds = xr.open_dataset(path)
    da = ds["PC"].isel(mode=slice(0, N_PR_MODES))
    df = da.to_pandas()
    df.index = pd.PeriodIndex(pd.to_datetime(df.index), freq="M")
    ds.close()
    return df

pr_train = load_pc_file(PR_TRAIN_FILE)
pr_test = load_pc_file(PR_TEST_FILE)

pr_pc = pd.concat([pr_train, pr_test]).sort_index()

if pr_pc.index.has_duplicates:
    raise ValueError("Duplicate months found in precipitation PC data.")

pr_pc.columns = [
    f"pr_pc{mode}" for mode in range(1, N_PR_MODES + 1)
]

print("Precipitation PC period:",
      pr_pc.index.min(), "—", pr_pc.index.max())
print("Precipitation PC shape:", pr_pc.shape)

Precipitation PC period: 1981-09 — 2025-12
Precipitation PC shape: (532, 3)


In [40]:
# 3. BUILD FEATURES FOR TARGET MONTH T
#    Features: SST PCs at T + seasonality.
#    Targets: precipitation PCs at T.

X_rows = []
y_rows = []
target_dates = []

for target_month in pr_pc.index:
    if target_month > TEST_END:
        continue

    # Skip months outside the specified train/test target periods.
    if target_month > TRAIN_END and target_month < TEST_START:
        continue

    features = {}

    for col in sst_pc.columns:
        features[col] = sst_pc.loc[target_month, col]

    # Seasonal features correspond to the target month.
    month = target_month.month
    features["month_sin"] = np.sin(2 * np.pi * month / 12)
    features["month_cos"] = np.cos(2 * np.pi * month / 12)

    target_values = pr_pc.loc[target_month].values

    if not np.all(np.isfinite(list(features.values()))):
        continue
    if not np.all(np.isfinite(target_values)):
        continue

    X_rows.append(features)
    y_rows.append(target_values)
    target_dates.append(target_month)

X = pd.DataFrame(X_rows, index=pd.PeriodIndex(
    target_dates, freq="M"
))
y = pd.DataFrame(
    y_rows,
    index=X.index,
    columns=[f"pr_pc{i}" for i in range(1, N_PR_MODES + 1)]
)

if X.index.has_duplicates:
    raise ValueError("Duplicate target months found.")

print("\nFull dataset:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target period:", X.index.min(), "—", X.index.max())


Full dataset:
X shape: (532, 8)
y shape: (532, 3)
Target period: 1981-09 — 2025-12


In [41]:
# 4. CHRONOLOGICAL TRAIN / TEST SPLIT

train_mask = X.index <= TRAIN_END
test_mask = (
    (X.index >= TEST_START)
    & (X.index <= TEST_END)
)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()
y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

if X_train.empty or X_test.empty:
    raise ValueError("Train or test set is empty. Check the dates and PC files.")

if not X_train.index.equals(y_train.index):
    raise ValueError("Train feature and target dates do not match.")

if not X_test.index.equals(y_test.index):
    raise ValueError("Test feature and target dates do not match.")

print("\nTrain:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Period:", X_train.index.min(), "—", X_train.index.max())

print("\nTest:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("Period:", X_test.index.min(), "—", X_test.index.max())


Train:
X_train: (472, 8)
y_train: (472, 3)
Period: 1981-09 — 2020-12

Test:
X_test: (60, 8)
y_test: (60, 3)
Period: 2021-01 — 2025-12


In [42]:
# 5. SAVE DATASETS

def save_with_date(df, filename):
    output = df.copy()
    output.insert(0, "target_month", output.index.astype(str))
    output.to_csv(OUTPUT_DIR / filename, index=False)

save_with_date(X_train, "X_train.csv")
save_with_date(X_test, "X_test.csv")
save_with_date(y_train, "y_train.csv")
save_with_date(y_test, "y_test.csv")

# Save NumPy arrays for direct model training.
np.savez_compressed(
    OUTPUT_DIR / "ml_dataset_arrays.npz",
    X_train=X_train.to_numpy(dtype=np.float64),
    X_test=X_test.to_numpy(dtype=np.float64),
    y_train=y_train.to_numpy(dtype=np.float64),
    y_test=y_test.to_numpy(dtype=np.float64),
    X_feature_names=np.array(X.columns),
    y_target_names=np.array(y.columns),
    train_dates=np.array(X_train.index.astype(str)),
    test_dates=np.array(X_test.index.astype(str)),
)

print(f"\nSaved dataset files to: {OUTPUT_DIR.resolve()}")

sst_ds.close()



Saved dataset files to: /home/26n5005/my project/ml_dataset_mean
